# 1장. Custom Middleware — Wrap-style: 동적 모델 선택

**Wrap-style Hook**(`@wrap_model_call`)은 LLM 호출 자체를 감싸(intercept) 실행 흐름을 동적으로 제어합니다.

이번 예시에서는 사용자의 **메시지 길이**에 따라 다른 모델을 선택합니다.

| 메시지 길이 | 선택 모델 | 이유 |
|:---|:---|:---|
| < 10자 | `gpt-5-nano` | 간단한 질문 → 가벼운 모델로 비용 절약 |
| 10~30자 | `gpt-5-mini` | 중간 복잡도 |
| > 30자 | `gpt-5` | 복잡하고 긴 질문 → 강력한 모델 |

> `request.override(model=new_model)` — 기존 요청 객체를 불변으로 유지하면서 모델만 교체한 새 요청 반환

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [ ]:
from langchain_openai import ChatOpenAI
from typing import Callable

from langchain.agents.middleware import wrap_model_call
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def dynamic_model_selector(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    # 최근 사용자의 입력 메시지 추출
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    # 길이에 따라 모델 선택
    if msg_len < 10:
        model_name = "gpt-5-nano"
    elif msg_len < 30:
        model_name = "gpt-5-mini"
    else:
        model_name = "gpt-5"

    print(f"🔁 메시지 길이: {msg_len}, 선택된 모델: {model_name}")

    # request.model을 새로운 모델로 교체
    new_model = ChatOpenAI(model_name=model_name)
    new_request = request.override(model=new_model)

    # 수정된 요청으로 LLM 호출
    return handler(new_request)


In [3]:
routing_agent = create_agent(
    model="gpt-5-mini", # 기본 모델로 생성했지만, 미들웨어가 알아서 교체해 줍니다.
    tools=[],
    middleware=[dynamic_model_selector],
)

routing_agent.invoke({"messages": [{"role": "user", "content": "안녕하세요"}]})
routing_agent.invoke({"messages": [{"role": "user", "content": "안녕하세요. 저는 Jay입니다."}]})
routing_agent.invoke({"messages": [{"role": "user", "content": "안녕하세요. 저는 Jay입니다. 인공지능 에이전트 아키텍처에 관해 궁금한 게 있어요."}]})


🔁 메시지 길이: 5, 선택된 모델: gpt-5-nano
🔁 메시지 길이: 17, 선택된 모델: gpt-5-mini
🔁 메시지 길이: 47, 선택된 모델: gpt-5


{'messages': [HumanMessage(content='안녕하세요. 저는 Jay입니다. 인공지능 에이전트 아키텍처에 관해 궁금한 게 있어요.', additional_kwargs={}, response_metadata={}, id='4f1b54f3-5221-487f-a8a8-1a9cc905c36b'),
  AIMessage(content='Jay님 반갑습니다! 인공지능 에이전트 아키텍처 전반을 간단히 잡아드리고, 필요하시면 특정 영역을 깊게 들어가요.\n\n에이전트의 핵심 개념\n- 목표를 달성하기 위해 환경을 관찰하고, 계획을 세우고, 도구를 사용해 행동하며, 결과를 바탕으로 다음 스텝을 조정하는 루프를 갖춘 LLM 기반 시스템입니다. 흔히 Sense–Think–Act(Observe–Plan/Reason–Act) 루프라고 부릅니다.\n\n필수 구성 요소\n- 입력/관찰: 사용자 메시지, 문서, 캘린더/이메일, 웹, 데이터베이스, 파일 등\n- 추론 엔진(LLM): 역할·시스템 프롬프트, 도구 호출(function/tool calling), 구조화 출력(JSON) 지원\n- 도구(액션): 검색/RAG, 코드 실행, 웹브라우저, DB/SQL, 외부 API, 워크플로 엔진\n- 상태와 메모리\n  - 작업 메모리: 현재 턴의 컨텍스트, 중간 계산 결과\n  - 단기/세션 메모리: 대화 이력, 최근 결론\n  - 장기 메모리: 벡터DB의 문서·사건(episodic), 지식베이스/그래프(semantic), 스킬/매크로(procedural)\n- 계획·제어 로직: ReAct(생각+행동 반복), Plan-Execute, Reflexion(자기평가), Tree/Graph-of-Thoughts, 논리적 스텝 제한, 예산·시간 한도\n- 오케스트레이션/상태기계: 에이전트 루프를 안전하게 이어주는 그래프/워크플로(예: LangGraph, Autogen, CrewAI, LlamaIndex Agents)\n- 관측·평가: 트레이싱, 코스트/지연 모니터링, 실패 리플레이, A